In [ ]:
import subprocess, sys, os, shutil

PYLIBS = "/kaggle/working/census_pylibs"
if not os.path.exists(PYLIBS) or True:  # force clean reinstall this run
    shutil.rmtree(PYLIBS, ignore_errors=True)
    os.makedirs(PYLIBS, exist_ok=True)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "--target", PYLIBS, "--no-deps",
        "cellxgene_census", "tiledbsoma", "somacore==1.0.29",
        "anndata==0.10.9", "natsort", "array_api_compat", "legacy-api-wrap",
        "exceptiongroup", "pyarrow-hotfix", "wrapt", "scverse-misc",
        "httpx", "httpcore", "h11", "anyio", "sniffio",
        "s3fs", "aiobotocore", "botocore",
        "aioitertools", "frozenlist", "multidict", "yarl",
        "async-timeout", "aiosignal", "aiohappyeyeballs", "aiohttp",
        "jmespath", "python-dateutil", "six", "urllib3", "idna",
        "certifi", "charset-normalizer", "attrs", "propcache", "fsspec",
    ], check=True)
sys.path.insert(0, PYLIBS)

# ---- everything below is the actual recheck script ----
from __future__ import annotations
import json
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd

IS_KAGGLE = Path("/kaggle").exists()

METADATA_H5AD_PATH = "/kaggle/input/datasets/ericdu847/metadata/metadata.h5ad" if IS_KAGGLE else "data/raw/metadata.h5ad"
RECHECK_CSV_PATH = (
    "/kaggle/input/datasets/ericdu847/patha-part2-recheck-sample-csv/pathA_part2_recheck_sample.csv"
    if IS_KAGGLE else "data/processed/pathA_part2_recheck_sample.csv"
)
OUTPUT_DIR = "/kaggle/working/part2_recheck_results" if IS_KAGGLE else "part2_recheck_results"

N_CELLS_PER_TYPE = 800


def log(m):
    print(f"[recheck] {m}", flush=True)


def safe_name(ct: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_-]+", "_", ct)[:80]


def load_decima_gene_set(metadata_path):
    import h5py
    f = h5py.File(metadata_path, "r")
    gene_id = f["var"]["gene_id"][:].astype(str)
    f.close()
    return gene_id.tolist()


def pull_one_cell_type(census, census_cell_type: str, gene_id_list):
    import cellxgene_census

    meta = census["census_data"]["homo_sapiens"].obs.read(
        value_filter=f"cell_type == '{census_cell_type}' and is_primary_data == True",
        column_names=["soma_joinid", "dataset_id"],
    ).concat().to_pandas()
    if len(meta) == 0:
        return None, {"n_cells_available": 0}
    best_dataset = meta["dataset_id"].value_counts().idxmax()
    pool = meta[meta.dataset_id == best_dataset]
    n = min(N_CELLS_PER_TYPE, len(pool))
    rng = np.random.RandomState(hash(census_cell_type) % (2**31))
    sample_ids = pool["soma_joinid"].sample(n=n, random_state=rng).tolist()

    adata = cellxgene_census.get_anndata(
        census, organism="Homo sapiens",
        obs_coords=sample_ids,
        X_name="raw",
    )
    return adata, {"n_cells_available": len(meta), "n_cells_pulled": adata.n_obs, "dataset_id": best_dataset}


def cpm_log1p(adata, gene_id_order):
    import scipy.sparse as sp
    X = adata.X
    if sp.issparse(X):
        X = X.toarray()
    lib = X.sum(axis=1, keepdims=True)
    lib[lib == 0] = 1
    cpm = X / lib * 1e6
    logx = np.log1p(cpm)
    var_ids = adata.var["feature_id"].values if "feature_id" in adata.var else adata.var_names.values
    idx_map = {g: i for i, g in enumerate(var_ids)}
    aligned = np.full((logx.shape[0], len(gene_id_order)), np.nan)
    n_found = 0
    for j, g in enumerate(gene_id_order):
        if g in idx_map:
            aligned[:, j] = logx[:, idx_map[g]]
            n_found += 1
    return aligned, n_found


def main():
    import cellxgene_census

    out = Path(OUTPUT_DIR)
    out.mkdir(parents=True, exist_ok=True)

    log(f"loading Decima gene set from {METADATA_H5AD_PATH}")
    gene_id_list = load_decima_gene_set(METADATA_H5AD_PATH)
    log(f"  {len(gene_id_list)} genes")

    log(f"loading recheck targets from {RECHECK_CSV_PATH}")
    targets = pd.read_csv(RECHECK_CSV_PATH)
    log(f"  {len(targets)} cell types to recheck")

    census = cellxgene_census.open_soma(census_version="2025-11-08")

    rows = []
    for i, row in targets.reset_index(drop=True).iterrows():
        decima_ct, census_ct = row["decima_cell_type"], row["census_cell_type"]
        t0 = time.time()
        try:
            adata, meta = pull_one_cell_type(census, census_ct, gene_id_list)
            if adata is None:
                log(f"  [{i+1}/{len(targets)}] {decima_ct}: no cells found, skipping")
                continue
            aligned, n_found = cpm_log1p(adata, gene_id_list)
            meta["n_genes_found"] = n_found
            n = aligned.shape[0]
            rng = np.random.RandomState(hash(decima_ct) % (2**31))
            idx = rng.permutation(n)
            out_df = pd.DataFrame({
                "mean_expr_full": np.nanmean(aligned, axis=0),
                "mean_expr_half_a": np.nanmean(aligned[idx[:n // 2]], axis=0),
                "mean_expr_half_b": np.nanmean(aligned[idx[n // 2:]], axis=0),
            })
            out_path = out / f"{safe_name(decima_ct)}.parquet"
            out_df.to_parquet(out_path)
            (out / f"{safe_name(decima_ct)}.meta.json").write_text(
                json.dumps({"cell_type": decima_ct, **meta}))
            log(f"  [{i+1}/{len(targets)}] {decima_ct}: {time.time()-t0:.1f}s, "
                f"{n_found}/{len(gene_id_list)} genes found")
            rows.append({"cell_type": decima_ct, "n_genes_found": n_found})
        except Exception as e:  # noqa: BLE001
            log(f"  [{i+1}/{len(targets)}] {decima_ct}: FAILED: {e}")

    census.close()

    if rows:
        summary = pd.DataFrame(rows)
        summary.to_csv(out / "_recheck_summary.csv", index=False)
        log("summary:")
        log(summary.to_string())
    log(f"recheck complete. Results in {out}/")


if __name__ == "__main__":
    main()